In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
import matplotlib.image as mpimg

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 7 &mdash; LTI Systems, Convolution and Correlation</h1></center>

Every geophysical recording is the Earth's response filtered by something else. The source wavelet, the instrument and the propagation path
all smear what we want to see. When that filtering is **linear and time-invariant (LTI)**, a single function, the impulse response, describes it
completely, and one operation, **convolution**, predicts the output for any input. This module builds convolution from the seismic and GPR
convolutional model, connects it to the Fourier transforms of Module 5 (convolution in time is multiplication in frequency), introduces its close
relative **crosscorrelation**, which is used for picking traveltimes and matching signals, and ends with the attempt to undo convolution,
**deconvolution**, and why that attempt is fragile.

[Linear time-invariant](https://en.wikipedia.org/wiki/Linear_time-invariant_theory) (LTI) systems are important in many branches of science and engineering (including geophysics!) because they can be completely characterized by the [impulse response](https://en.wikipedia.org/wiki/Impulse_response), which is how a system responds to an impulsive $\delta(t)$ function input. One example of a linear system with an impulse response would be if you gave a small impulsive push to a friend sitting on a swing. What would the response of the system be? Damped oscillation back and forth until eventually coming to rest (where the duration of the period depends on the swing length and the overall amplitude decay depends on the swing's effective damping constant). This time series effectively represents the impulse response of the swing system.

Examples of LTI systems include:

* Spring-mass systems
* Electrical circuits
* 1D Seismic/GPR convolutional modeling 

In fact, [convolution](https://en.wikipedia.org/wiki/Convolution)  and its counterpart [correlation](https://en.wikipedia.org/wiki/Cross-correlation) are two LTI operations that represent two key concepts that you will learn and apply in this course.  Thus, it is worth spending some time on developing your intuition of what this is.  Let us first discuss the geophysical example of convolutional modeling, which is one way to think about forward modeling both seismic and GPR data.

# Outline
- [7.1 Convolutional Modeling of Seismic and GPR Data](#7.1-Convolutional-Modeling-of-Seismic-and-GPR-Data)
- [7.2 Two Key Properties of LTI Systems](#7.2-Two-Key-Properties-of-LTI-Systems)
- [7.3 The Convolution Integral](#7.3-The-Convolution-Integral)
  - [7.3.1 Small numerical examples](#7.3.1-Small-numerical-examples)
  - [7.3.2 Evaluating the convolution integral as a movie](#7.3.2-Evaluating-the-convolution-integral-as-a-movie)
  - [7.3.3 Example 1: a spike and a Gaussian](#7.3.3-Example-1:-a-spike-and-a-Gaussian)
- [7.4 Convolution in the Fourier Domain](#7.4-Convolution-in-the-Fourier-Domain)
  - [7.4.1 Example 2: convolution of two boxcars](#7.4.1-Example-2:-convolution-of-two-boxcars)
  - [7.4.2 Commutativity](#7.4.2-Commutativity)
  - [7.4.3 Computational efficiency](#7.4.3-Computational-efficiency)
- [7.5 Physical LTI Systems](#7.5-Physical-LTI-Systems)
  - [7.5.1 GPR acquisition system](#7.5.1-GPR-acquisition-system)
  - [7.5.2 Human hearing](#7.5.2-Human-hearing)
- [7.6 Convolution in Action](#7.6-Convolution-in-Action)
  - [7.6.1 Convolution with a δ-function](#7.6.1-Convolution-with-a-δ-function)
  - [7.6.2 Convolution with a pair of δ-functions](#7.6.2-Convolution-with-a-pair-of-δ-functions)
  - [7.6.3 Two Gaussian functions](#7.6.3-Two-Gaussian-functions)
  - [7.6.4 The algebra of convolutions](#7.6.4-The-algebra-of-convolutions)
  - [7.6.5 The derivative theorem](#7.6.5-The-derivative-theorem)
  - [7.6.6 The convolution derivative theorem](#7.6.6-The-convolution-derivative-theorem)
  - [7.6.7 Parseval's and Rayleigh's theorems](#7.6.7-Parseval's-and-Rayleigh's-theorems)
- [7.7 2D Convolution](#7.7-2D-Convolution)
  - [7.7.1 Spatial-domain example](#7.7.1-Spatial-domain-example)
  - [7.7.2 Wavenumber-domain example](#7.7.2-Wavenumber-domain-example)
- [7.8 Crosscorrelation](#7.8-Crosscorrelation)
  - [7.8.1 Crosscorrelation properties](#7.8.1-Crosscorrelation-properties)
  - [7.8.2 The difference between convolution and correlation](#7.8.2-The-difference-between-convolution-and-correlation)
  - [7.8.3 The autocorrelation theorem](#7.8.3-The-autocorrelation-theorem)
- [7.9 Deconvolution](#7.9-Deconvolution)
  - [7.9.1 Deconvolution in practice](#7.9.1-Deconvolution-in-practice)
- [7.10 Summary](#7.10-Summary)
- [Microquestions](#Microquestions)

<a name="7.1"></a>
## 7.1 Convolutional Modeling of Seismic and GPR Data

The Earth's P-wave acoustic impedance ($I_p=\rho V_p $) is commonly represented in terms of 'reflectivity', $r(t)\approx \frac{1}{2}\frac{d}{dt}\ln I_p = \frac{V_p}{4}\frac{d}{dz}\ln I_p$, where $t = 2z/V_p$ is two-way traveltime. At a sharp boundary between layers of impedance $I_1$ (above) and $I_2$ (below), this becomes the reflection coefficient $R = (I_2 - I_1)/(I_2 + I_1)$.  One straightforward approach to modeling a 1D (or higher dimension) seismic section is to **convolve** a **seismic wavelet**, $w(t)$, with an **Earth reflectivity sequence**, $r(t)$.  The **output** of this operation, $s(t)$, is approximately the **seismic data trace** you would record between a co-located source and receiver pair.  

Mathematically, this operation is denoted as:  

$$ s(t) = w(t)\ast r(t)+n(t) \tag{1}$$

where $\ast$ represents the convolution operation here between $w(t)$ and $r(t)$, and $n(t)$ is some additive noise.  Figure 7.1 illustrates the approach for a simple two-interface model.

In [ ]:
C = DSP_COLORS
# (a) two-layer impedance model in depth
z = np.linspace(800, 1300, 1001)
sand = (z >= 1000) & (z < 1100)
Vp = np.where(sand, 3000., 2400.); rho = np.where(sand, 2.20, 2.30)
I = rho*Vp/1000                                       # impedance in (km/s)(g/cm^3)
# (b) reflection coefficients in two-way time t = 2z/V
I1, I2 = 2.30*2400, 2.20*3000
R = (I2 - I1)/(I2 + I1)
t1 = 2*1000/2400; t2 = t1 + 2*100/3000
dt = 0.001; t = np.arange(0.70, 1.0305, dt)
r = np.zeros_like(t); r[np.argmin(abs(t - t1))] = R; r[np.argmin(abs(t - t2))] = -R
# (c) trace s = w * r with a 30 Hz Ricker wavelet (zero phase, centered)
tw = np.arange(-0.1, 0.1 + dt/2, dt)
w = (1 - 2*(np.pi*30*tw)**2)*np.exp(-(np.pi*30*tw)**2)
s = np.convolve(r, w, mode='same')
fig, axs = plt.subplots(1, 4, figsize=(13, 5.2), gridspec_kw=dict(width_ratios=[1.1, 1, 1, 1]))
axs[0].plot(I, z, color=C['continental_crust'], lw=2.5)
axs[0].set_ylim(1300, 800); axs[0].set_xlim(5.2, 6.9)
axs[0].set_xlabel(r'$I_p=\rho V_p$ [(km/s)(g/cm$^3$)]'); axs[0].set_ylabel('depth $z$ [m]')
axs[0].set_title('(a) Impedance model')
axs[0].text(5.55, 1050, 'sand', va='center', fontsize=11); axs[0].text(5.65, 900, 'shale', va='center', fontsize=11)
axs[1].hlines(t[r != 0], 0, r[r != 0], color=C['input'], lw=2.5)
axs[1].plot(r[r != 0], t[r != 0], 'o', color=C['input'])
axs[1].axvline(0, color='k', lw=0.8)
for tv, rv in zip(t[r != 0], r[r != 0]):
    axs[1].text(rv + (0.012 if rv > 0 else -0.012), tv, f'{rv:+.3f}', va='center', ha='left' if rv > 0 else 'right', fontsize=10)
axs[1].set_xlim(-0.21, 0.21); axs[1].set_title('(b) Reflectivity $r(t)$')
for ax, sg, ttl in ((axs[2], 1, '(c) $s=w\\ast r$, SEG normal'), (axs[3], -1, '(d) Reversed polarity')):
    ax.plot(sg*s, t, color='k', lw=1.5)
    ax.fill_betweenx(t, 0, sg*s, where=sg*s > 0, color=C['output'], alpha=0.6, lw=0)
    ax.axvline(0, color='k', lw=0.8); ax.set_xlim(-0.12, 0.12); ax.set_title(ttl)
for ax in axs[1:]:
    ax.set_ylim(1.03, 0.70); ax.set_ylabel('two-way time $t$ [s]'); ax.set_xlabel('amplitude')
plt.tight_layout(); plt.show()

<b>Figure 7.1.</b> The seismic convolutional model as an LTI system. (a) A 100 m sand layer (impedance 6.60) within shale (5.52). (b) The reflectivity in two-way time: $R = +0.0891$ at the top of the sand ($t = 0.8333$ s) and $-0.0891$ at its base ($t = 0.900$ s). (c) The trace $s = w\ast r$ for a 30 Hz Ricker wavelet in the SEG normal polarity convention, in which a positive reflection coefficient gives a peak (shaded). (d) The same trace in the reversed (European) polarity convention. <i>(Python-generated.)</i>

<a name="7.2"></a>
## 7.2 Two Key Properties of LTI Systems

As per the name, a LTI system (here defined by $H[\cdot]$) has two key necessary ingredients:

* [Linearity](https://en.wikipedia.org/wiki/Linear_system): if $[x_1(t),y_1(t)]$ and $[x_2(t),y_2(t)]$ are two input-output pairs (i.e., $y_1(t) = H[x_1(t)]$ and $y_2(t) = H[x_2(t)]$ for some LTI operator $H$), then:

$$ax_1(t)+bx_2(t) \rightarrow ay_1(t)+by_2(t) \tag{2} $$

This principle simply states that you may superimpose two different input signals and the resulting output signal will be the same superposition as if you treated them separately and then stacked the outputs. In the convolutional modeling example above, this would mean that you could: (1) convolve each component of the reflection sequence with the wavelet individually and then stack the result; or (2) convolve the wavelet with the entire reflectivity sequence all at once. Following either scenario will generate the same result.

* [Time-invariance](https://en.wikipedia.org/wiki/Time-invariant_system): If $x(t)\rightarrow y(t)$ then

$$ x(t-t_0) \rightarrow y(t-t_0), \quad {\rm for\ all}\ t_0 \tag{3}$$

This implies that delaying or advancing the operation by a constant $t_0$ has no effect on how the system responds. (If you were to push your friend on the swing today or tomorrow in an equal manner, would you expect there to be any difference in the resulting motion?)  In the seismic convolutional modeling example above, this means that if you shifted the input reflectivity sequence $r(t)$ to $r(t-t_0)$, you would expect the output seismic trace to be similarly shifted from $y(t)$ to $y(t-t_0)$.

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as eigenfunctions.</b> Feed an LTI system the complex exponential $x(t) = e^{i\omega t}$ of
Module 3. The convolution integral, equation (4) below, gives $y(t) = \int h(\tau)\,e^{i\omega(t-\tau)}\,d\tau = \widehat{H}(\omega)\,e^{i\omega t}$: the
output is the <i>same</i> exponential, scaled by the number $\widehat{H}(\omega)$, the Fourier transform of $h$ from Module 5. Complex exponentials are
therefore the eigenfunctions of every LTI system, and this one line is why the convolution theorem of Section 7.4 must be true.
</div>

<a name="7.3"></a>
## 7.3 The Convolution Integral

One of the most important aspects of an LTI system is that the input-output relationship is characterized by a [convolution](https://en.wikipedia.org/wiki/Convolution) operation alluded to - but not explicitly defined - above.  The convolution integral is written in the following way:

$$ y(t) = x(t) \ast h(t) = \int_{-\infty}^{\infty} x(\tau)\,h(t-\tau)\, \mathrm{d}\tau \tag{4} $$

where $x(t), y(t)$ and $h(t)$ are the input, output and the response of the LTI system. In equation (4) we see that a dummy integration variable $\tau$ has been introduced.  You'll also notice that one function, $h(t-\tau)$, is **time reversed** relative to the other variable. 

Here is a short pseudo-code to think about the mathematical operation in equation 4:

    For each $t$ in the output space:
        For each $\tau$ in the input space:
            Multiply $x(\tau)$ with time-reversed $h(t-\tau)$
            Accumulate result into output $y(t)$
            
Let's now look at a couple of examples.

<a name="7.3.1"></a>
### 7.3.1 Small numerical examples

The NumPy function `np.convolve` evaluates the discrete version of equation (4):

In [ ]:
a = np.array([1, 2, 3])
b = np.array([3, 2, 1])
c = np.array([1, 0, -1])
print('a convolved with a:', np.convolve(a, a))
print('b convolved with b:', np.convolve(b, b))
print('a convolved with b:', np.convolve(a, b))
print('b convolved with a:', np.convolve(b, a))
print('b convolved with c:', np.convolve(b, c))

Notice that $a\ast b$ and $b\ast a$ are identical (convolution commutes; Section 7.4.2), and that each output has $3 + 3 - 1 = 5$ samples.

<a name="7.3.2"></a>
### 7.3.2 Evaluating the convolution integral as a movie

It is an instructive exercise to plot $x(t)$ and a time reversed $h(t)$ on two different pieces of paper.  Initially line them up with no shift applied (i.e., $\tau=0$).  Mentally multiply two functions together and sum the result (i.e., the total area under the curve) to get your output $y(0)$. You can then shift one paper by a $\tau$ units and repeat this exercise. Eventually if you do this for all different values of $\tau$, you can build up the convolution result $y(t)$.

Figure 7.2 does exactly this for a one-sided exponential input, $x(\tau) = e^{-\tau}u(\tau)$ (blue), and the boxcar $h = \Pi_{1/2}$ of half-width 0.5 s (green). Each frame is one value of $t$ in equation (4). Watch three things. The box $h(t-\tau)$ slides to the right as $t$ increases (it is symmetric, so the time reversal does not change its shape). The shaded overlap $x(\tau)h(t-\tau)$ grows while the box covers the jump in $x$, then shrinks as the box moves into the decaying tail. And the area of that overlap, printed in the title, is the value of $y(t)$ plotted in the lower panel: it peaks at $y = 1 - e^{-1} = 0.632$ at $t = 0.5$ s, when the box exactly covers $0 \le \tau \le 1$. (Example 2 below convolves a boxcar with itself.)

In [ ]:
C = DSP_COLORS
n_frames = 30
tk = -1 + 0.125*np.arange(n_frames)                 # t = -1 ... 2.625 s (frame 12 is the peak, t = 0.5 s)
tau = np.linspace(-2, 4, 6001)
x = np.where(tau >= 0, np.exp(-tau), 0.)            # x(tau) = exp(-tau) u(tau)
box = lambda u: (np.abs(u) <= 0.5).astype(float)    # h = boxcar of half-width 0.5 s
y_an = lambda t: np.where(t < -0.5, 0., np.where(t < 0.5, 1 - np.exp(-(t + 0.5)), np.exp(-(t - 0.5)) - np.exp(-(t + 0.5))))
tauf = np.linspace(-2, 4, 60001)                    # fine grid for the printed areas
xf = np.where(tauf >= 0, np.exp(-tauf), 0.)
ys = np.array([np.trapezoid(xf*box(t - tauf), tauf) for t in tk])
tt = np.linspace(-1.5, 3.5, 501)
fig, (axa, axb) = plt.subplots(2, 1, figsize=(8.2, 4.6), sharex=True)
axa.plot(tau, x, color=C['input'], lw=2, label=r'$x(\tau)=e^{-\tau}u(\tau)$')
hline, = axa.plot([], [], color=C['impulse_resp'], lw=2, label=r'$h(t-\tau)$')
axa.set_xlim(-1.5, 3.5); axa.set_ylim(-0.05, 1.25); axa.set_ylabel('amplitude'); axa.legend(loc='upper right', fontsize=10)
axb.plot(tt, y_an(tt), color=C['neutral'], lw=4, alpha=0.3, label='analytic $y(t)$')
yline, = axb.plot([], [], 'o-', color=C['output'], ms=5, lw=1.2, label=r'$y(t)=\int x(\tau)h(t-\tau)\,d\tau$')
axb.set_ylim(-0.05, 0.75); axb.set_xlabel(r'$\tau$ (top), $t$ (bottom) [s]'); axb.set_ylabel('$y(t)$'); axb.legend(loc='upper right', fontsize=10)
state = {'fill': None}
title = fig.suptitle('', fontsize=13)

def update(k):
    t = tk[k]
    hline.set_data(tau, box(t - tau))
    if state['fill'] is not None:
        state['fill'].remove()
    state['fill'] = axa.fill_between(tau, 0, x*box(t - tau), color=C['output'], alpha=0.35, lw=0)
    yline.set_data(tk[:k + 1], ys[:k + 1])
    title.set_text(f't = {t:.3f} s,  y(t) = shaded area = {ys[k]:.3f}')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=150, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 7.2.</b> Convolution as flip and slide, 30 frames from $t = -1$ to $2.625$ s. (a) The input $x(\tau) = e^{-\tau}u(\tau)$ (blue; also the impulse response of an RC circuit with $RC = 1$ s), the shifted boxcar $h(t-\tau)$ (green) and their product (shaded), whose area is $y(t)$. (b) The output $y(t)$ built up frame by frame (red) on top of the analytic result (gray). <i>(Python-generated.)</i>

<a name="7.3.3"></a>
### 7.3.3 Example 1: a spike and a Gaussian

Let's look at the (numerical) convolution of a single input delta function spike $f(t)$ and a supposed Gaussian impulse response of the system $g(t)$ that generates our output $f(t)\ast g(t)$.  We'll repeat this example four times where we apply a different delay to the input spike. The Gaussian is centered on sample 0 (zero phase), so each output pulse lands exactly on its spike.

In [ ]:
C = DSP_COLORS
nt = 201                                   # samples, index 100 = lag 0
lag = np.arange(nt) - nt//2
klag = np.arange(-20, 21)
g = np.exp(-klag**2/(2*4.0**2))            # zero-phase Gaussian kernel, sigma = 4 samples, centered on lag 0

def spike_panels(rows, xlim, ylim=(-1.25, 1.25)):
    """Left: spikes f (blue) and kernel g (green); right: y = f * g (red). One row per input."""
    fig, axs = plt.subplots(len(rows), 2, figsize=(12, 2.35*len(rows)), sharex=True)
    for i, f in enumerate(rows):
        y = np.convolve(f, g, mode='same')
        nz = f != 0
        axs[i, 0].vlines(lag[nz], 0, f[nz], color=C['input'], lw=2)
        axs[i, 0].plot(lag[nz], f[nz], 'o', color=C['input'], label=r'$f(\tau)$')
        axs[i, 0].plot(klag, g, color=C['impulse_resp'], lw=2, label=r'$g(\tau)=g(-\tau)$')
        axs[i, 1].plot(lag, y, color=C['output'], lw=2, label=r'$y=f\ast g$')
        axs[i, 0].set_title(f'({chr(97 + 2*i)}) Input spikes and kernel', fontsize=12)
        axs[i, 1].set_title(f'({chr(98 + 2*i)}) Output $y = f\\ast g$', fontsize=12)
        for ax in axs[i]:
            ax.axhline(0, color='k', lw=0.6); ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_ylabel('amplitude')
    axs[0, 0].legend(loc='lower left', fontsize=9, ncol=2); axs[0, 1].legend(loc='lower left', fontsize=9)
    for ax in axs[-1]:
        ax.set_xlabel('sample (lag)')
    plt.tight_layout(); plt.show()

rows = []
for idx in (10, 30, 50, 70):               # one spike at four delays (lags -90, -70, -50, -30)
    f = np.zeros(nt); f[idx] = 1; rows.append(f)
spike_panels(rows, xlim=(-100, 25))

<b>Figure 7.3.</b> Convolving a single spike with a Gaussian. Left panels (a), (c), (e), (g): the spike $f(\tau)$ (blue) at four different delays and the zero-phase Gaussian kernel $g(\tau)$ (green, $\sigma = 4$ samples), which equals its own time reverse. Right panels (b), (d), (f), (h): the output $y = f\ast g$ (red) is a copy of the Gaussian centered on each spike. <i>(Python-generated.)</i>

Let's now look at the convolution of a Gaussian with a sequence of pulses that are spread out like the earth reflectivity model described above:

In [ ]:
rows = []
for pos, amp in (([20], [1]), ([20, 40], [1, -1]), ([20, 40, 60], [1, -1, 0.5]), ([20, 25, 30, 35], [1, -1, 0.5, -0.5])):
    f = np.zeros(nt); f[pos] = amp; rows.append(f)
spike_panels(rows, xlim=(-100, 25))

<b>Figure 7.4.</b> As in Figure 7.3, but with one, two, three and four spikes. The outputs in (b), (d) and (f) show one, two and three separate pulses, one per spike in (a), (c) and (e). In (g) the four spikes are only 5 samples apart, close compared with the width of the Gaussian, so the pulses in (h) overlap and interfere constructively and destructively: the output has 4 extrema, with amplitudes +0.67, -0.38, -0.12, -0.34, instead of four clean copies of the kernel with amplitudes $+1, -1, +0.5, -0.5$. <i>(Python-generated.)</i>

<a name="7.4"></a>
## 7.4 Convolution in the Fourier Domain

An important property is that **convolution** in the time domain corresponds to **multiplication** in the Fourier transform domain:

$$y(t) = x(t) \ast h(t) \quad\Longleftrightarrow\quad \widehat{Y}(\omega) = \widehat{X}(\omega)\,\widehat{H}(\omega) \tag{5}$$

<div class="alert alert-info">
<b>Proof of the convolution theorem:</b> The Fourier transform of $y(t)=x(t)\ast h(t)$ is:

$$ 
\begin{eqnarray}
\widehat{Y}(\omega) &=& \mathcal{F} \left[ x(t)\ast h(t) \right] \\
\, &=& \mathcal{F} \left[ \int_{-\infty}^{\infty} x(\tau)\,h(t-\tau) \,\mathrm{d}\tau \right] \\
\, &=& \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} x(\tau)\,h(t-\tau)\, \mathrm{e}^{-i\omega t} \,\mathrm{d}\tau \, \mathrm{d}t\\
\end{eqnarray}\tag{6}
$$

Let's now assert that $t=u+\tau$ where $u$ is the new integration variable and $\tau$ is a constant shift.  Thus, we may write $\mathrm{d}t = \mathrm{d}u$.  Inserting this into the equation above yields:

$$
\begin{eqnarray}
\widehat{Y}(\omega) &=&  \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} x(\tau)\,h(u)\, \,\mathrm{e}^{-i\omega (u+\tau)}\,\mathrm{d}\tau \,\mathrm{d}u\\
\, &=&  \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} x(\tau)\,h(u)\, \mathrm{e}^{-i\omega \tau}\,\mathrm{e}^{-i\omega u}\,\mathrm{d}\tau \,\mathrm{d}u\\
\,&=& \left[\int_{-\infty}^{\infty} x(\tau)\,\mathrm{e}^{-i\omega \tau}\,\mathrm{d}\tau\right]
\left[
\int_{-\infty}^{\infty} h(u)\,\mathrm{e}^{-i\omega u}\,\mathrm{d}u
\right]\\
\,&=& \widehat{X}(\omega)\,\widehat{H}(\omega)
\end{eqnarray}\tag{7}
$$
</div>

Thus, the Fourier transform of a convolution is the product of the Fourier transforms of the two functions.  This is one of the most important properties of convolution.

<div class="alert alert-info">
<b>Synthesis &mdash; convolution and multiplication.</b> Module 5's shift theorem (a delay by $a$ multiplies the spectrum by $e^{-i\omega a}$) is the
special case $h = \delta(t-a)$ of the convolution theorem (Section 7.6.1). Its modulation property, multiplying by $e^{i\omega_0 t}$ in time, is the dual
statement: multiplication in one domain is convolution in the other, up to the factor $1/2\pi$ of our convention (equation 31).
</div>

<a name="7.4.1"></a>
### 7.4.1 Example 2: convolution of two boxcars

This example looks at the (auto)convolution of two boxcar functions of equal width.  You might recognize the result ... a triangle function!  Recall from Module 5 that, for the boxcar $\Pi_T$ of half-width $T$ and the normalized sinc,

$$ \Pi_T(t) \Longleftrightarrow 2T\,\mathrm{sinc}\left(\frac{\omega T}{\pi}\right) \tag{8} $$

Therefore, by the convolution theorem:

$$\Pi_T(t) \ast \Pi_T(t) \Longleftrightarrow 4T^2\,\mathrm{sinc}^2\left(\frac{\omega T}{\pi}\right)\tag{9} $$ 

You'll also recall that a $\mathrm{sinc}^2$ function is the Fourier Transform of the triangle function. Therefore, you probably wouldn't be surprised to hear that the autoconvolution of a boxcar function is the triangle function! This is illustrated in Figure 7.5.

In [ ]:
C = DSP_COLORS
nt = 100
t = np.arange(nt)/nt
tau = t - 0.495                                     # time axis centered for mode='same'
x1 = np.zeros(nt); x1[20:30] = 0.5;  h1 = np.zeros(nt); h1[70:80] = 0.5
x2 = np.zeros(nt); x2[10:40] = 0.5;  h2 = np.zeros(nt); h2[55:75] = 0.5
fig, axs = plt.subplots(2, 2, figsize=(12, 6))
for row, (x, h, nx, nh) in enumerate(((x1, h1, '$x_1$', '$h_1$'), (x2, h2, '$x_2$', '$h_2$'))):
    y = np.convolve(x, h, mode='same')
    axs[row, 0].plot(tau, x, color=C['input'], lw=2, label=nx + '$(t)$')
    axs[row, 0].plot(tau, h, color=C['impulse_resp'], lw=2, ls='--', label=nh + '$(t)$')
    axs[row, 0].set_ylim(-0.05, 0.75); axs[row, 0].legend(loc='upper left', fontsize=10)
    axs[row, 0].set_title(f'({"ac"[row]}) Inputs ' + nx + ' and ' + nh, fontsize=12)
    axs[row, 1].plot(tau, y, color=C['output'], lw=2)
    axs[row, 1].set_ylim(0, 1.15*y.max())
    axs[row, 1].set_title(f'({"bd"[row]}) ' + nx + r'$\ast$' + nh, fontsize=12)
    for ax in axs[row]:
        ax.set_xlim(-0.5, 0.5); ax.set_xlabel('time [s]'); ax.set_ylabel('amplitude')
plt.tight_layout(); plt.show()

<b>Figure 7.5.</b> (a) Two boxcars of equal width (0.1 s, height 0.5) and (b) their convolution, a triangle whose peak, $10 \times 0.5 \times 0.5 = 2.5$, occurs when the two boxes overlap completely. (c) Boxcars of unequal width (0.3 s and 0.2 s) and (d) their convolution, a trapezoid. Its flat top (0.1 s wide, the difference of the widths) corresponds to shifts for which the shorter box lies entirely inside the longer one. <i>(Python-generated.)</i>

Thus, to compute the autoconvolution of the boxcar function you can do either one of two things:

1. Compute the convolution of $y(t) = h(t)\ast x(t)$ in the physical domain using a convolution program.
2. Compute the Fourier transforms of $h(t)$ and $x(t)$, multiply the results (i.e., $\widehat{Y}(\omega)=\widehat{X}(\omega)\widehat{H}(\omega)$), and then take the inverse Fourier transform of $\widehat{Y}(\omega)$ to get the desired result $y(t)$.

<a name="7.4.2"></a>
### 7.4.2 Commutativity

Let's look at proving the following:

$$f(t) \ast g(t) = g(t) \ast f(t) \tag{10}$$

Let's prove it analytically:

<div class="alert alert-info">
<b>Proof that convolution commutes:</b> The definition of convolution is given by:
$$f(t)\ast g(t) = \int_{-\infty}^{\infty} f(\tau)g(t-\tau)d\tau \tag{11}$$

However, we may perform a substitution of $t^\prime=t-\tau$, $d\tau=-dt^\prime$ and $\tau=t-t^\prime$ in the above equation.

$$ f(t)\ast g(t) =
- \int_{t+\infty}^{t-\infty} f(t-t^\prime)g(t^\prime)dt^\prime 
\tag{12}$$

However, care must be paid to the integration limits.  First, we note that as $\tau\rightarrow-\infty$ we get $t^\prime=t+\infty=+\infty$.  Similarly, as $\tau\rightarrow +\infty$ we get $t^\prime=t-\infty=-\infty$.  This is why we have had to change the integration limits in equation (12). 

The next step is that we reverse the integration limits, which changes the sign of the integral.  Thus, we may write:

$$ f(t)\ast g(t) =
- \int_{+\infty}^{-\infty} f(t-t^\prime)g(t^\prime)dt^\prime 
= \int_{-\infty}^{+\infty}g(t^\prime) f(t-t^\prime)dt^\prime 
=
g(t) \ast f(t)
\tag{13} $$

Thus, the convolution operator is commutative. This should be expected from the Fourier convolution theorem which states: 

$$f(t)\ast g(t) \Leftrightarrow \widehat{F}(\omega)\widehat{G}(\omega)=\widehat{G}(\omega)\widehat{F}(\omega) \Leftrightarrow g(t)\ast f(t) \tag{14} $$
</div>

Figure 7.12(b) in Section 7.8 confirms this numerically.

<a name="7.4.3"></a>
### 7.4.3 Computational efficiency

From a computational perspective, as the number of samples in a time series gets "large" it becomes more efficient to compute a (fast) Fourier Transform, perform Fourier domain multiplication and then take the inverse Fourier transform.  The [computational complexity](https://en.wikipedia.org/wiki/Computational_complexity_theory) of this operation is $\mathcal{O}(N\log N)$ rather than $\mathcal{O}(N^2)$ of performing convolution directly in the time domain. Figure 7.6 counts operations for a simple model: $N^2$ multiply–adds for direct convolution of two length-$N$ signals, against about $3N\log_2 N + N$ for the FFT route (two forward FFTs and one inverse FFT of about $N\log_2 N$ operations each, plus $N$ products).

In [ ]:
C = DSP_COLORS
N = 2.0**np.arange(3, 17)                       # 8 ... 65536 samples
direct = N**2                                   # multiply-adds for direct convolution
fftway = 3*N*np.log2(N) + N                     # two forward FFTs + one inverse (~N log2 N each) + N products
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.6))
axa.loglog(N, direct, 'o-', color=C['time_domain'], label='direct: $N^2$')
axa.loglog(N, fftway, 's-', color=C['freq_domain'], label=r'FFT route: $3N\log_2 N + N$')
axa.set_xlabel('number of samples $N$'); axa.set_ylabel('operations'); axa.legend(loc='upper left', fontsize=10)
axa.set_title('(a) Cost of convolving two length-$N$ signals', fontsize=12)
axb.loglog(N, direct/fftway, 'o-', color='k')
axb.axhline(1, color=C['neutral'], ls=':')
axb.set_xlabel('number of samples $N$'); axb.set_ylabel('direct / FFT route')
axb.set_title('(b) Speed-up of the FFT route', fontsize=12)
for ax in (axa, axb):
    ax.set_xlim(6, 9e4)
plt.tight_layout(); plt.show()

<b>Figure 7.6.</b> (a) Operation counts for convolving two length-$N$ signals directly ($N^2$, blue) and through the FFT ($3N\log_2 N + N$, red). The FFT route is already cheaper at $N = 16$ (208 operations against 256). (b) The ratio of the two counts grows almost linearly with $N$; it is 33 at $N = 1024$. <i>(Python-generated.)</i>

<a name="7.5"></a>
## 7.5 Physical LTI Systems

<a name="7.5.1"></a>
### 7.5.1 GPR acquisition system

Consider that you are recording a geophysical signal (e.g., GPR wave) denoted by $x(t)$.  The instrument (GPR receiving antenna) you are using has a natural response, $h(t)$.  The recorded GPR data you measure, $y(t)$, will be affected by the instrument you are using through convolution: 

$$y(t)=x(t)\ast h(t) \tag{15}$$  

Thus, recorded data $y(t)$ will not be the ''true'' response $x(t)$ unless: (1) you 
have a ''perfectly non-distorting'' instrument (i.e., $h(t)=\delta(t)$); or (2) you account for  the instrument's response through a [deconvolution](https://en.wikipedia.org/wiki/Deconvolution) procedure to recover  $x(t)$ (that is, assuming you know $h(t)$):

$$ x(t) = \mathcal{F}^{-1}\left[ \frac{\widehat{Y}(\omega)}{\widehat{H}(\omega)} \right] =  \mathcal{F}^{-1}\left[ \frac{\mathcal{F}[y(t)]} {\mathcal{F}[h(t)]} \right] \tag{16} $$

<div class="alert alert-warning">
<b>Question:</b> What are the potential problems or challenges with a deconvolutional approach?
</div>

<div class="alert alert-info">
<b>Synthesis &mdash; the RC circuit as an LTI system.</b> The RC low-pass circuit of PHGN 200 is an LTI system. Its capacitor voltage obeys
$RC\,\frac{dy}{dt} + y = x(t)$, and solving it with the MATH 225 integrating factor gives $y(t) = \int_0^\infty \frac{1}{RC}e^{-\tau/RC}\,x(t-\tau)\,d\tau$, a
convolution with $h(t) = \frac{1}{RC}e^{-t/RC}u(t)$: the one-sided exponential sliding through Figure 7.2. Its spectrum $\widehat{H}(\omega) =
1/(1 + i\omega RC)$ has the familiar PHGN 200 corner frequency $f_c = 1/(2\pi RC)$ (Microquestion 7.2).
</div>

<a name="7.5.2"></a>
### 7.5.2 Human hearing

There is a far-field acoustic source $x(t)$ in your presence.  Your hearing system has two inputs $h_L(t)$ and $h_R(t)$ (i.e., your ears!).  Unfortunately, they are an imperfect sensing instrument and thus what you actually ''hear'' is a convolution product:

$$ y(t) = \left[ h_L(t)+h_R(t)\right]\ast x(t) \tag{17} $$

Hearing aids with responses $a_L(t)$ and $a_R(t)$ can be introduced to improve your response!!  This leads to a more complicated LTI model:

$$ y(t) = \left[ a_L(t) \ast h_L(t)+a_R(t) \ast h_R(t)\right]\ast x(t) \tag{18} $$

Figure 7.7 illustrates this model.

<center><img src="Fig/2.3-EARS.png" width="300"></center>

<b>Figure 7.7.</b> The ears plus hearing aids as an LTI system. An acoustic source produces a signal $x(t)$ that propagates through the air and is detected by the listener, whose hearing system has two detectors, the left and right ears with responses $h_L(t)$ and $h_R(t)$. The brain receives the convolution of the ears' responses with the signal. With hearing aids, each ear's response is further convolved with the hearing-aid response $a_L(t)$ or $a_R(t)$, which can differ between the two ears (equation 18). <i>(Drawing by Gavin Wilson (former GPGN 404 teaching assistant).)</i>

<a name="7.6"></a>
## 7.6 Convolution in Action

<a name="7.6.1"></a>
### 7.6.1 Convolution with a δ-function

Convolution of a function $f(t)$ with a $\delta$-function shifted by $a$ unit gives

$$  f(t) \ast \delta(t-a) = \int_{-\infty}^{\infty} f(\tau)\,\delta(t-a-\tau)\,\mathrm{d}\tau = f(t-a) \tag{19} $$

by virtue of the properties of the $\delta$-function.  We may write this symbolically as

$$ f(t) \ast \delta (t-a) = f(t-a) \tag{20} $$

Applying the convolution theorem to this result yields the Fourier shift theorem of Module 5:

$$ f(t) \Leftrightarrow \widehat{F}(\omega); \quad \delta(t-a) \Leftrightarrow \mathrm{e}^{-i \omega a} \tag{21}
$$

which yields the following:

$$ f(t-a) = f(t) \ast \delta (t-a) \Leftrightarrow \widehat{F}(\omega)\mathrm{e}^{-i\omega a} \tag{22} $$

which is the multiplication of the two signals' Fourier spectra! The interpretation of this is that the shifting of a time series by $a$ units leads to the similar Fourier spectrum that has a phase shift of ${\rm e}^{-i\omega a}$ applied to it.

<a name="7.6.2"></a>
### 7.6.2 Convolution with a pair of δ-functions

Based on our previous exploration of the Fourier transform of $\delta$-functions, we can identify the following:


$$ \left[\delta(t-a)+\delta(t+a)\right] \Leftrightarrow 2\,\mathrm{cos}\,(\omega a) \tag{23} $$

with a similar result holding if the delta functions were defined in the Fourier domain.  The convolution of the expression in equation (23) with a function $f(t)$ is given by

$$ \left[\delta(t-a)+\delta(t+a)\right] \ast f(t) \Leftrightarrow 2\,\mathrm{cos}\,(\omega a) \widehat{F}(\omega) \tag{24} $$

which is again the product of the signals' respective Fourier transforms.

<a name="7.6.3"></a>
### 7.6.3 Two Gaussian functions

An interesting example that comes up quite frequently in probability theory (including geophysical inverse problems  when modeling probability density functions) is the convolution of two Gaussian functions. Say we have two Gaussian functions of width $a$ and $b$, $f(t)=\mathrm{e}^{-t^2/a^2}$ and $g(t)=\mathrm{e}^{-t^2/b^2}$ respectively.  In the Fourier domain we know that the respective Fourier transforms are given by:

$$\widehat{F}(\omega) = a\sqrt{\pi}\,\mathrm{e}^{-a^2\omega^2/4} \tag{25} $$

and

$$\widehat{G}(\omega) = b\sqrt{\pi}\,\mathrm{e}^{-b^2\omega^2/4} \tag{26} $$

Thus, according to the Fourier domain convolution theorem, we may write:

$$ 
\begin{eqnarray}
f(t) \ast g(t) &=& \mathcal{F}^{-1}\left[\widehat{F}(\omega)\cdot \widehat{G}(\omega)\right]\tag{27} \\
\,&=& \mathcal{F}^{-1}\left[
ab\pi \mathrm{e}^{-(a^2+b^2)\omega^2/4}
\right] \tag{28}\\
\, &=& \frac{a b \sqrt{\pi} }{\sqrt{a^2+b^2}}\mathrm{e}^{-t^2/(a^2+b^2)} \tag{29}
\end{eqnarray}
$$

Importantly, the convolution of two Gaussians with widths $a$ and $b$ **is also a Gaussian of width $\sqrt{a^2+b^2}$.**  Let's test this out with a numerical example (Figure 7.8).

In [ ]:
C = DSP_COLORS
dt = 0.002
t = np.arange(-1, 1 + dt/2, dt)
fig, axs = plt.subplots(2, 2, figsize=(12, 6))
for row, (a, b) in enumerate(((0.05, 0.05), (0.05, 0.15))):
    f = np.exp(-t**2/a**2); g = np.exp(-t**2/b**2)
    y_num = np.convolve(f, g, mode='same')*dt                       # discrete approximation of the integral
    y_an = a*b*np.sqrt(np.pi)/np.sqrt(a**2 + b**2)*np.exp(-t**2/(a**2 + b**2))
    axs[row, 0].plot(t, f, color=C['input'], lw=2, label=f'$f=e^{{-t^2/a^2}}$, a = {a} s')
    axs[row, 0].plot(t, g, color=C['impulse_resp'], lw=2, ls='--', label=f'$g=e^{{-t^2/b^2}}$, b = {b} s')
    axs[row, 0].set_ylim(0, 1.35); axs[row, 0].legend(loc='upper right', fontsize=9)
    axs[row, 0].set_title(f'({"ac"[row]}) Two Gaussians', fontsize=12)
    axs[row, 1].plot(t, y_num, color=C['output'], lw=4, alpha=0.5, label='numerical $f\\ast g$')
    axs[row, 1].plot(t, y_an, 'k--', lw=1.5, label=f'equation: width $\\sqrt{{a^2+b^2}}$ = {np.hypot(a, b):.3f} s')
    axs[row, 1].set_ylim(0, 1.75*y_an.max()); axs[row, 1].legend(loc='upper right', fontsize=9)
    axs[row, 1].set_title(f'({"bd"[row]}) Their convolution', fontsize=12)
    for ax in axs[row]:
        ax.set_xlim(-0.5, 0.5); ax.set_xlabel('time [s]'); ax.set_ylabel('amplitude')
plt.tight_layout(); plt.show()

<b>Figure 7.8.</b> The convolution of two Gaussians. (a) Two Gaussians of equal width, $a = b = 0.05$ s, and (b) their convolution computed numerically (thick red) and from equation (29) (dashed): a Gaussian of width $\sqrt{a^2+b^2} = 0.071$ s. (c) Unequal widths, $a = 0.05$ s and $b = 0.15$ s, and (d) their convolution, of width 0.158 s. Convolving with a Gaussian always broadens a signal. <i>(Python-generated.)</i>

One interesting observation is that because the resulting function after the convolution is broader, convolution with a Gaussian filter is a **low-pass filtering** operation. One can use this approach to remove high-frequency or high-wavenumber noise in a data set.

<a name="7.6.4"></a>
### 7.6.4 The algebra of convolutions

One has to take care when applying more complex convolution operators.  For example, generally speaking:

<div class="alert alert-warning">
<b>Common misconception:</b> Convolution and multiplication do not associate with each other. In general,
$$ \left[a(t) \ast b(t)\right] \cdot c(t) \neq a(t)\ast\left[b(t)\cdot c(t)\right] \tag{30}$$
</div>

One can use the properties of the convolution operation to derive some truly convoluted relationships:

$$ 
\left[a(t)\ast b(t)\right]\cdot \left[c(t)\ast d(t)\right]
\Leftrightarrow
\frac{1}{2\pi}\left[\widehat{A}(\omega) \cdot \widehat{B}(\omega)\right]\ast \left[\widehat{C}(\omega)\cdot \widehat{D}(\omega)\right]  \tag{31}
$$

The factor $1/2\pi$ comes from our Fourier convention (Module 5): multiplication in time corresponds to convolution in frequency divided by $2\pi$.

<a name="7.6.5"></a>
### 7.6.5 The derivative theorem

If $f(t)\Leftrightarrow\widehat{F}(\omega)$ and $g(t) =\frac{\mathrm{d}f(t)}{\mathrm{d}t}$  then 

$$ \widehat{G}(\omega) = \mathcal{F}[g(t)] = \mathcal{F}\left[\frac{\mathrm{d}f(t)}{\mathrm{d}t}\right] = i\omega \widehat{F}(\omega) \tag{32} $$

which is sometimes written symbolically as a Fourier Transform pair:

$$\frac{\mathrm{d}}{\mathrm{d}t} \Leftrightarrow i\omega \tag{33}$$

Note that we have so far been examining the Fourier Transforms of functions; however, this is the first instance when we have encountered the Fourier Transform of an **operator** (i.e., $\frac{\mathrm{d}}{\mathrm{d}t}$).

<div class="alert alert-info">
<b>Proof of the derivative theorem:</b> Let's begin with our definition of the Fourier Transform of $f(t)$ and its derivative:

$$ \widehat{F}(\omega) = \int_{-\infty}^{\infty} f(t)e^{-i\omega t} dt \tag{34} $$

and

$$ \widehat{G}(\omega) = \int_{-\infty}^{\infty} \frac{d f(t)}{dt} e^{-i\omega t} dt \tag{35}$$

Let's look at the second integral and apply integration by parts using $u=e^{-i\omega t}$, $dv=\frac{df}{dt} dt$, $v=f$ and $du = -i\omega e^{-i\omega t} dt $.

Thus, 

$$\int_{-\infty}^{\infty} u dv = \left. uv\right|_{-\infty}^{\infty} - \int_{-\infty}^{\infty} v du \tag{36}$$

may be written:

$$\widehat{G}(\omega) =\int_{-\infty}^{\infty} \frac{df(t)}{dt} e^{-i\omega t}dt = \left. f (t) e^{-i\omega t}\right|_{-\infty}^{\infty} + \int_{-\infty}^{\infty} f(t) i\omega e^{-i\omega t} dt \tag{37}$$

The boundary term must vanish at both $\pm\infty$ for $f(t)$ to be an admissible function for the Fourier transform.  Thus, we may write:

$$\widehat{G}(\omega) = \int_{-\infty}^{\infty} \frac{df(t)}{dt} e^{-i\omega t}dt = i\omega \int_{-\infty}^{\infty} f(t) e^{-i\omega t} dt = i\omega \widehat{F}(\omega) \tag{38}$$

which is equation (32).
</div>

Note that if we defined our Fourier transform with the opposite convention (i.e., $e^{i\omega t}$ instead of $e^{-i\omega t}$) then the result would be $\frac{\mathrm{d}}{\mathrm{d}t} \Leftrightarrow - i\omega$, which is commonly found in mathematical physics textbooks.

The Fourier derivative theorem also extends to higher derivatives:

$$ \frac{\mathrm{d}^nf(t)}{\mathrm{d}t^n} \Longleftrightarrow \left(i\omega\right)^n \widehat{F}(\omega) \tag{39} $$

This is a very important thing to remember because we often take derivatives of functions but do not always think about how this affects the Fourier spectrum - especially the DC component that is set to zero.

Note that this also holds for partial differential equations.  For example, if we have the 1D acoustic wave equation

$$\frac{\partial^2 u}{\partial x^2} = \frac{1}{c^2}\frac{\partial^2 u}{\partial t^2} \tag{40} $$

we can use the fact that $\frac{\partial}{\partial t} \Leftrightarrow i\omega$ and $\frac{\partial}{\partial x} \Leftrightarrow ik_x$ to immediately write:

$$ (ik_x)^2 = \frac{(i\omega)^2}{c^2} \tag{41}$$

or 

$$ k_x = \pm \frac{\omega}{c} \tag{42}$$

which you may recognize as a dispersion relationship linking wavenumbers and frequencies!

<div class="alert alert-info">
<b>Synthesis &mdash; waves in the Fourier domain.</b> The dispersion relation $k_x = \pm\omega/c$ is the PHGN 200 wave-on-a-string result,
$c = \sqrt{T/\mu}$, written in Fourier form, and it is the plane-wave relation used for seismic P and S waves in POTE1 (GPGN 328). The derivative theorem
turned a PDE into algebra. The same substitution, $\partial/\partial x \Leftrightarrow ik_x$, is what made the wavenumber filtering of potential-field maps
in Module 6 possible.
</div>

<a name="7.6.6"></a>
### 7.6.6 The convolution derivative theorem

The following is true:

$$ \frac{\mathrm{d}}{\mathrm{d}t}
\left[
f(t)\ast g(t)
\right] =
\frac{\mathrm{d}f(t)}{\mathrm{d}t}\ast g(t)
=
f(t)\ast \frac{\mathrm{d}g(t)}{\mathrm{d}t}  \tag{43}
$$

**Extension Problem:** Prove that this is correct.

<a name="7.6.7"></a>
### 7.6.7 Parseval's and Rayleigh's theorems

The following is true:

$$\int_{-\infty}^{\infty} f(t) \overline{g(t)} \mathrm{d}t = \frac{1}{2\pi}\int_{-\infty}^{\infty} \widehat{F}(\omega) \overline{\widehat{G}(\omega)} \mathrm{d}\omega  \tag{44}$$

Two particular cases of interest are Rayleigh's theorem

$$\int_{-\infty}^{\infty} \left|f(t)\right|^2 \mathrm{d}t = \frac{1}{2\pi}\int_{-\infty}^{\infty}\left| \widehat{F}(\omega)\right|^2 \mathrm{d}\omega \tag{45}$$

and the average power of a periodic signal of period $T$, represented by the Fourier series of Module 4 with coefficients $A_k$ and $B_k$:

$$\frac{1}{T}\int_{-T/2}^{T/2} \left|f(t)\right|^2 \mathrm{d}t = \frac{A_0^2}{4} + \frac{1}{2}\sum_{k=1}^{\infty} \left(A_k^2+B_k^2\right) \tag{46}$$

The factor $1/2\pi$ in equations (44) and (45) belongs to our Fourier convention (Module 5); with frequency $f$ in hertz instead of $\omega$ it disappears.

<a name="7.7"></a>
## 7.7 2D Convolution

Like its 1D cousin, **2D convolution** may be expressed in the following way:

$$ g(x,y) = h(x,y) \ast f(x,y) = f(x,y) \ast h(x,y) \tag{47} $$

In integral notation, 2D convolution may be expressed as

$$ g(x,y) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(\tau_x,\tau_y) h(x-\tau_x,y-\tau_y) \,\mathrm{d}\tau_x\,\mathrm{d}\tau_y \tag{48} $$

Similarly to the 1D scenario, one can develop an expression for the **2D Convolution Theorem**:

$$ f(x,y) \ast h(x,y) \Leftrightarrow \widehat{F}(k_x,k_y)\,\widehat{H}(k_x,k_y) \tag{49}$$

<a name="7.7.1"></a>
### 7.7.1 Spatial-domain example

Let's look at an example of a 2D convolution involving a picture of a brick wall, $f(x,y)$ (the photograph of Module 6, in grayscale and reduced to 260 × 260 pixels), and a 2D Gaussian filter $h(x,y)$ with a standard deviation of $\sigma = 3$ pixels. The result in Figure 7.9 was computed directly in the space domain with equation (48).

In [ ]:
from scipy import ndimage
C = DSP_COLORS
f2 = mpimg.imread('Fig/2.3_BRICKS.jpg').astype(float).mean(axis=2)[::2, ::2]   # grayscale, 260 x 260 pixels
k = np.arange(-20, 21); KX, KY = np.meshgrid(k, k)
h2 = np.exp(-(KX**2 + KY**2)/(2*3.0**2)); h2 /= h2.sum()                    # 2D Gaussian, sigma = 3 px, 41 x 41 support
g2 = ndimage.convolve(f2, h2, mode='wrap')                                 # direct 2D convolution, equation (45)
fig, axs = plt.subplots(1, 3, figsize=(12, 4.2))
axs[0].imshow(f2, cmap='gray'); axs[0].set_title('(a) Input $f(x,y)$', fontsize=12)
im = axs[1].imshow(h2, cmap='viridis', extent=[-20.5, 20.5, 20.5, -20.5]); axs[1].set_title(r'(b) Filter $h(x,y)$, $\sigma=3$ px', fontsize=12)
fig.colorbar(im, ax=axs[1], shrink=0.8)
axs[2].imshow(g2, cmap='gray'); axs[2].set_title(r'(c) Output $g=h\ast f$', fontsize=12)
for ax in axs:
    ax.set_xlabel('$x$ [pixels]'); ax.set_ylabel('$y$ [pixels]'); ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 7.9.</b> A 2D convolution computed in the space domain. (a) The input image $f(x,y)$. (b) The 2D Gaussian filter $h(x,y)$ ($\sigma = 3$ pixels, 41 × 41 samples, normalized to unit sum). (c) The output $g = h\ast f$: the edges of the bricks are blurred, because the filter averages each pixel with its neighbors. <i>(Python-generated from a photograph from <a href="https://commons.wikimedia.org/wiki/File:Solna_Brick_wall_Stretcher_bond_variation1.jpg">Wikimedia Commons</a>.)</i>

<a name="7.7.2"></a>
### 7.7.2 Wavenumber-domain example

Now let's recompute the previous example using the **2D convolution theorem**. This evaluation (1) applies 2D Fourier transforms to $f$ and $h$ to obtain $\widehat{F}$ and $\widehat{H}$; (2) multiplies these two functions together to obtain $\widehat{G} = \widehat{F}\widehat{H}$; and then (3) applies an inverse Fourier transform to recover $g$. Figure 7.10 shows the three spectra, using the spatial frequencies $u = k_x/2\pi$ and $v = k_y/2\pi$ in cycles per pixel.

In [ ]:
Kf = np.zeros_like(f2); Kf[:41, :41] = h2; Kf = np.roll(Kf, (-20, -20), axis=(0, 1))   # filter centered on pixel (0, 0)
F2, H2 = np.fft.fft2(f2), np.fft.fft2(Kf)
G2 = F2*H2                                                                     # 2D convolution theorem, equation (46)
g2_fft = np.real(np.fft.ifft2(G2))
print(f'Largest difference between inverse FFT of F*H and the direct result of Figure 7.9: {np.max(np.abs(g2_fft - g2)):.1e}')
uax = np.fft.fftshift(np.fft.fftfreq(f2.shape[1])); ext = [uax[0], uax[-1], uax[-1], uax[0]]
sh = lambda A: np.fft.fftshift(A)
lo = np.log10(np.abs(F2)).min()
fig, axs = plt.subplots(1, 3, figsize=(12, 4.2))
axs[0].imshow(np.log10(np.abs(sh(F2))), cmap='magma', extent=ext); axs[0].set_title(r'(a) $\log_{10}|\widehat{F}|$', fontsize=12)
axs[1].imshow(np.abs(sh(H2)), cmap='viridis', extent=ext); axs[1].set_title(r'(b) $|\widehat{H}|$', fontsize=12)
axs[2].imshow(np.log10(np.abs(sh(G2)) + 1e-12), cmap='magma', extent=ext, vmin=lo); axs[2].set_title(r'(c) $\log_{10}|\widehat{G}| = \log_{10}|\widehat{F}\widehat{H}|$', fontsize=12)
for ax in axs:
    ax.set_xlabel('$u$ [cycles/pixel]'); ax.set_ylabel('$v$ [cycles/pixel]'); ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 7.10.</b> The 2D convolution of Figure 7.9 computed with the convolution theorem, in the wavenumber domain. (a) $\log_{10}|\widehat{F}|$ of the brick image; the bright horizontal and vertical lines come from the regular rows and joints of the bricks. (b) $|\widehat{H}|$, a Gaussian that passes low wavenumbers and suppresses high ones. (c) $\log_{10}|\widehat{G}|$, the product of (a) and (b). The inverse transform of $\widehat{G}$ is the blurred image of Figure 7.9(c): both routes treat the image as periodic, and they agree to within 2e-12 (printed above the figure). <i>(Python-generated from a photograph from <a href="https://commons.wikimedia.org/wiki/File:Solna_Brick_wall_Stretcher_bond_variation1.jpg">Wikimedia Commons</a>.)</i>

<a name="7.8"></a>
## 7.8 Crosscorrelation

An important operation related to convolution is [cross correlation](https://en.wikipedia.org/wiki/Cross-correlation).  This operation is denoted by the $\star$ operator and is closely related to convolution. The integral for cross correlation is given by:

$$y(\tau) = f (\tau) \star g(\tau) = \int _{-\infty }^{\infty }\overline{f(t)}\ g(t+\tau )\,dt \tag{50}$$

where $\overline{f(t)}$ is the complex conjugate and unlike in convolution $g(t+\tau )$ is not time reversed. The $\tau$ axis may be considered as the **correlation lag** or **time delay** between the two signals $f$ and $g$. However, one can switch the places of $t$ and $\tau$ in equation (50) without any change in mathematical meaning.

In NumPy, `np.correlate(a, v, mode='full')` conjugates and slides its <i>second</i> argument, so in the notation of equation (50) the correlation $f\star g$ is computed by

    np.correlate(g, f, mode='full')

(note the order). There is again a *mode='same'* option, which returns a result of the same length as the longer input.

Figure 7.11 shows four examples computed with *mode='full'*:

In [ ]:
C = DSP_COLORS
n = 101
t = np.arange(n)/n                                  # 0 ... 1 s
lag = (np.arange(2*n - 1) - (n - 1))/n              # mode='full' lags, -0.99 ... 0.99 s
xs_ = [0.5*np.exp(-(t - 0.5)**2/(2*0.02**2)), 0.25*np.exp(-(t - 0.5)**2/(2*0.08**2)),
       0.5*np.exp(-(t - 0.3)**2/(2*0.02**2)), 1.0*np.exp(-(t - 0.5)**2/(2*0.02**2))]
h0 = np.zeros(n); h0[20:30] = 0.01*np.arange(10)**2
hs_ = [h0, np.exp(-(t - 0.5)**2/(2*0.02**2)), np.exp(-(t - 0.5)**2/(2*0.02**2)), 0.2*np.ones(n)]
fig, axs = plt.subplots(4, 2, figsize=(12, 9))
for i, (x, h) in enumerate(zip(xs_, hs_)):
    xh = np.correlate(h, x, mode='full')            # = (x star h)(tau) of equation (47): numpy slides its SECOND argument
    axs[i, 0].plot(t, x, color=C['input'], lw=2, label='$x(t)$')
    axs[i, 0].plot(t, h, color=C['impulse_resp'], lw=2, ls='--', label='$h(t)$')
    axs[i, 0].set_xlim(0, 1); axs[i, 0].set_ylim(-0.05, 1.15); axs[i, 0].set_xlabel('time [s]')
    axs[i, 0].set_title(f'({chr(97 + 2*i)}) Signals $x(t)$ and $h(t)$', fontsize=12); axs[i, 0].legend(loc='upper right', fontsize=9)
    axs[i, 1].plot(lag, xh, color=C['output'], lw=2)
    axs[i, 1].set_xlim(-1, 1); axs[i, 1].set_ylim(min(0, xh.min()) - 0.05*np.abs(xh).max(), 1.2*xh.max()); axs[i, 1].set_xlabel(r'lag $\tau$ [s]')
    axs[i, 1].set_title(f'({chr(98 + 2*i)}) Crosscorrelation $(x\\star h)(\\tau)$', fontsize=12)
    for ax in axs[i]:
        ax.set_ylabel('amplitude')
plt.tight_layout(); plt.show()

<b>Figure 7.11.</b> Crosscorrelation of four pairs of signals. Left panels (a), (c), (e), (g): the signals $x(t)$ (blue) and $h(t)$ (green, dashed). Right panels (b), (d), (f), (h): the crosscorrelation $(x\star h)(\tau)$ of equation (50), computed as <code>np.correlate(h, x, mode='full')</code>. In (f) the peak is at $\tau = +0.20$ s, the delay of $h$ (centered at 0.5 s) relative to $x$ (centered at 0.3 s). Crosscorrelation is not commutative: swapping $x$ and $h$ reverses the lag axis. <i>(Python-generated.)</i>

<div class="alert alert-warning">
<b>Note:</b> The lag axis in Figure 7.11 runs from −1 s to 1 s because these examples use <code>mode='full'</code>, not <code>mode='same'</code> as in the
convolution examples. For arrays of lengths $N$ and $M$, <code>mode='full'</code> returns all $N + M - 1$ lags (here $2N - 1$, from $-(N-1)$ to $N-1$ samples),
while <code>mode='same'</code> returns $\max(N, M)$ samples centered on the full result. Our signals are defined on 0–1 s, so the delays between them can be as
large as ±1 s.
</div>

<a name="7.8.1"></a>
### 7.8.1 Crosscorrelation properties

The correlation integral has the following important properties:

(1) The cross-correlation of $f(t)$ and $g(t)$ is equivalent to the **convolution** of $\overline{f(-t)}$ and $g(t)$.
    
$$f(t)\star g(t) = \overline{f(-t)}\ast g(t) \tag{51} $$

(2) If $f(t)$ is a [Hermitian function](https://en.wikipedia.org/wiki/Hermitian_function) (i.e., has the property of $\overline{f(t)}=f(-t)$) then the following holds:

$$f(t) \star g(t)  = f (t) \ast g(t)\tag{52}$$

(3) If $f(t)$ and $g(t)$ are both Hermitian functions, then 

$$ f(t) \star g(t) = g(t)\star f(t) \tag{53}$$

(4) The cross-correlation of two functions [i.e., $h(t) = f(t) \star g(t)$] satisfies the following in the Fourier domain:

$$ \widehat{H}(\omega) =\mathcal{F}[h(t)] = \mathcal{F}[f(t)\star g(t)] =  \mathcal{F}[\overline{f(-t)}\ast g(t)] = \overline{\widehat{F}(\omega)} \widehat{G}(\omega) \tag{54} $$

<a name="7.8.2"></a>
### 7.8.2 The difference between convolution and correlation

Let's now look at an example showing the difference between convolution and crosscorrelation. In Figure 7.12, panel (b) overlays the convolutions $x\ast h$ and $h\ast x$, which are identical because convolution commutes (Section 7.4.2). Panel (c) overlays the crosscorrelations $y_1 = x\star h$ and $y_2 = h\star x$: here $y_2(\tau) = y_1(-\tau)$, as equation (51) predicts.

In [ ]:
C = DSP_COLORS
n = 201
t = np.arange(n)/n
x = 0.5*np.exp(-(t - 0.5)**2/(2*0.02**2))
h = np.zeros(n); h[20:70] = np.arange(50)/100
tc = np.arange(2*n - 1)/n                           # time axis of the full convolution
lag = (np.arange(2*n - 1) - (n - 1))/n
fig, axs = plt.subplots(3, 1, figsize=(10, 8.4))
axs[0].plot(t, x, color=C['input'], lw=2, label='$x(t)$'); axs[0].plot(t, h, color=C['impulse_resp'], lw=2, ls='--', label='$h(t)$')
axs[0].set_xlim(0, 1); axs[0].set_xlabel('time [s]'); axs[0].legend(loc='upper right', fontsize=10)
axs[0].set_title('(a) Signals $x(t)$ and $h(t)$', fontsize=12)
axs[1].plot(tc, np.convolve(x, h), color=C['output'], lw=5, alpha=0.45, label=r'$x\ast h$')
axs[1].plot(tc, np.convolve(h, x), 'k--', lw=1.5, label=r'$h\ast x$')
axs[1].set_xlim(0, 2); axs[1].set_xlabel('time [s]'); axs[1].legend(loc='upper right', fontsize=10)
axs[1].set_title(r'(b) Convolutions $x\ast h$ and $h\ast x$: identical', fontsize=12)
axs[2].plot(lag, np.correlate(h, x, mode='full'), color=C['output'], lw=2, label=r'$y_1 = x\star h$')
axs[2].plot(lag, np.correlate(x, h, mode='full'), color=C['phase'], lw=2, ls='--', label=r'$y_2 = h\star x$')
axs[2].set_xlim(-1, 1); axs[2].set_xlabel(r'lag $\tau$ [s]'); axs[2].legend(loc='upper right', fontsize=10)
axs[2].set_title(r'(c) Crosscorrelations: $y_2(\tau) = y_1(-\tau)$', fontsize=12)
for ax in axs:
    ax.set_ylabel('amplitude')
plt.tight_layout(); plt.show()

<b>Figure 7.12.</b> Convolution versus crosscorrelation. (a) A Gaussian $x(t)$ (blue) and a ramp $h(t)$ (green, dashed). (b) The convolutions $x\ast h$ (thick red) and $h\ast x$ (dashed black) lie on top of each other. (c) The crosscorrelations $y_1 = x\star h$ (red) and $y_2 = h\star x$ (purple, dashed) are time-reversed copies of each other. <i>(Python-generated.)</i>

<a name="7.8.3"></a>
### 7.8.3 The autocorrelation theorem

The **autocorrelation** of a function $f(t)$, here denoted by $a_f(t) = f(t) \star f(t)$, has the following relationship:

$$ \widehat{A_f}(\omega) = \overline{\widehat{F}(\omega)}\widehat{F}(\omega) = \left|\widehat{F}(\omega)\right|^2 \tag{55} $$

<div class="alert alert-info">
<b>Proof of the autocorrelation theorem:</b> Let's start with the definition of cross-correlation:

$$
a_f(t) = \int_{-\infty}^{\infty} \overline{f(t^\prime) }\,f(t+t^\prime)\, \mathrm{d}t^\prime \tag{56}
$$

Applying a Fourier Transforms to both sides

$$
\widehat{A_f}(\omega) =  \int_{-\infty}^{\infty}  \int_{-\infty}^{\infty} \overline{f(t^\prime)}\,f(t+t^\prime)\,\mathrm{e}^{-i\omega t} \mathrm{d}t^\prime \mathrm{d}t \tag{57}
$$

Let $\tau = t+t^\prime$. Then if $t^\prime$ is held constant, $\mathrm{d}t=\mathrm{d}\tau$, 

$$
\widehat{A_f}(\omega) =  \int_{-\infty}^{\infty}  \int_{-\infty}^{\infty}\overline{f(t^\prime)}\,f(\tau)\,\mathrm{e}^{-i\omega(\tau-t^\prime)} \mathrm{d}t^\prime \mathrm{d}\tau \tag{58}
$$

which can be separated 

$$
\widehat{A_f}(\omega) =  
\int_{-\infty}^{\infty}  \overline{f(t^\prime)} \,\mathrm{e}^{i\omega t^\prime}\,\mathrm{d}t^\prime
\cdot
\int_{-\infty}^{\infty} f(\tau)\,\mathrm{e}^{-i\omega \tau} \, \mathrm{d}\tau  \tag{59}
$$

which leads to 

$$ \widehat{A_f}(\omega) = \overline{\widehat{F}(\omega)}\widehat{F}(\omega) = \left|\widehat{F}(\omega)\right|^2 \tag{60}$$
</div>

In words, the Fourier transform of the **autocorrelation** is the function's **power spectrum**. Figure 7.13 checks this numerically.

In [ ]:
C = DSP_COLORS
n = 201
t = np.arange(n)/n
h = np.zeros(n); h[40:70] = np.arange(30)/40
ac = np.correlate(h, h, mode='full')                 # autocorrelation, 2n - 1 lags
lag = (np.arange(2*n - 1) - (n - 1))/n
nf = 2*n - 1                                         # same FFT length for (b) and (d)
f = np.fft.fftshift(np.fft.fftfreq(nf, 1/n))
Hm = np.fft.fftshift(np.abs(np.fft.fft(h, nf)))
Am = np.fft.fftshift(np.abs(np.fft.fft(ac)))
fig, axs = plt.subplots(2, 2, figsize=(12, 6))
axs[0, 0].plot(t, h, color=C['time_domain'], lw=2); axs[0, 0].set_xlim(0, 1); axs[0, 0].set_xlabel('time [s]')
axs[0, 0].set_title('(a) $h(t)$', fontsize=12)
axs[0, 1].plot(f, Hm, color=C['freq_domain'], lw=2); axs[0, 1].set_xlabel('frequency [Hz]'); axs[0, 1].set_ylabel(r'$|\widehat{H}(f)|$')
axs[0, 1].set_title('(b) Magnitude spectrum', fontsize=12)
axs[1, 0].plot(lag, ac, color=C['time_domain'], lw=2); axs[1, 0].set_xlim(-1, 1); axs[1, 0].set_xlabel(r'lag $\tau$ [s]')
axs[1, 0].set_title(r'(c) Autocorrelation $(h\star h)(\tau)$', fontsize=12)
axs[1, 1].plot(f, Am, color=C['freq_domain'], lw=4, alpha=0.5, label=r'$|\mathcal{F}[h\star h]|$')
axs[1, 1].plot(f, Hm**2, 'k--', lw=1.5, label=r'$|\widehat{H}(f)|^2$')
axs[1, 1].set_xlabel('frequency [Hz]'); axs[1, 1].set_ylabel('power'); axs[1, 1].legend(loc='upper right', fontsize=10)
axs[1, 1].set_title('(d) Spectrum of the autocorrelation', fontsize=12)
for ax in axs[:, 0]:
    ax.set_ylabel('amplitude')
for ax in axs[:, 1]:
    ax.set_xlim(-30, 30)
plt.tight_layout(); plt.show()

<b>Figure 7.13.</b> The autocorrelation theorem. (a) A ramp $h(t)$ and (b) its magnitude spectrum $|\widehat{H}(f)|$. (c) Its autocorrelation $(h\star h)(\tau)$, which is symmetric about zero lag and peaks there. (d) The magnitude of the Fourier transform of (c) (thick red) and $|\widehat{H}(f)|^2$ (dashed) are identical, as equation (55) states. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; measuring a delay.</b> Crosscorrelation is how a lag is measured. The correlation of a recorded GPR or seismic trace with the
source pulse peaks at the traveltime, and the autocorrelation theorem says the peak's sharpness is set by the pulse's power spectrum
$|\widehat{X}(\omega)|^2$. A broadband source gives a narrow autocorrelation and so a precise traveltime: the time–bandwidth trade-off of Module 5's
scaling theorem (Microquestion 7.3).
</div>

<a name="7.9"></a>
## 7.9 Deconvolution

I was also interested in the abilities of Scipy in performing deconvolution.  Recall that if $y(t) = x(t)\ast h(t)$ then this operation performs the equivalent of

$$x(t) = \mathcal{F}^{-1}\left[\frac{\widehat{Y}(\omega)}{\widehat{H}(\omega)}\right] \tag{61}$$

However, it does not compute it directly from the Fourier spectra.  Rather, it aims to find a **polynomial expression** that adequately approximates $h(t)$ and then effectively uses polynomial division to remove this influence of $h(t)$ from $y(t)$.  

This is analogous to the polynomial multiplication and division that you may have done in earlier courses. For example, if $x(t)=t^2+1$ and $h(t)=2t+7$, then $y(t)=x(t)\ast h(t)$ is the **convolution of these two polynomials**.  Let these vectors represent the polynomial coefficients: $u=[1,0,1]$ and $v=[2,7]$.  The resulting **discrete convolution** of $u$ and $v$ is $[2,7,2,7]$ or 

$$y(t) = 2t^3+7t^2+2t+7 \tag{62}$$

which is equivalent to **polynomial multiplication** of the two functions $x(t)$ and $h(t)$. Let's verify this numerically:

In [ ]:
u = [1, 0, 1]          # x(t) = t^2 + 1
v = [2, 7]             # h(t) = 2t + 7
aa = np.convolve(u, v, mode='full')
print('u:', u)
print('v:', v)
print('The convolution of u and v is:', aa)

The **deconvolution** of $h(t)$ from $y(t)$ is then equivalent to **polynomial division**:

$$x(t) = \frac{2t^3+7t^2+2t+7}{2t+7} = \frac{(t^2+1)(2t+7)}{2t+7} = t^2+1 \tag{63}$$

This approach works great if you can find a **stable** polynomial approximation for $h(t)$.  

In [ ]:
from scipy import signal
recovered, remainder = signal.deconvolve(aa, v)
print('The deconvolution of aa by v is:', recovered)
print('The remainder is:', remainder)

Such an inverse does not always exist; Module 14 shows when a polynomial inverse is stable. If we change the final 7 into a 6, $2t+7$ no longer divides the numerator exactly:

$$x(t) = \frac{2t^3+7t^2+2t+6}{2t+7} \tag{64}$$

In [ ]:
aanew = [2, 7, 2, 6]
recovered, remainder = signal.deconvolve(aanew, v)
print('The deconvolution of aanew by v is:', recovered)
print('The remainder is:', remainder)

The quotient is still $t^2+1$, but the remainder, $-1$, is the part that polynomial division could not remove.

Let's explore an example where convolution and then deconvolution is being applied using the following two function calls:

* *y = np.convolve(x,h,mode='same')* 
* *x, remainder = signal.deconvolve(y, h)* (from <code>scipy</code>)

In [ ]:
from scipy import signal
C = DSP_COLORS
nt = 300
random_noise_level = 0.001     # small random perturbation of the filter
DC_term = 0.01                 # try DC_term = 0 and rerun this cell!
rng = np.random.default_rng(404)                     # seeded, so the result is reproducible
tline1 = np.arange(nt); tline2 = np.arange(nt//6)
sig = np.repeat([0., 0., 1., 0., 0.], nt//5)         # box-like signal
gauss = np.exp(-((tline2 - 25.)/4)**2) + random_noise_level*np.abs(rng.standard_normal(nt//6)) + DC_term
filtered = np.convolve(sig, gauss, mode='same')/np.sum(gauss)
deconv, remainder = signal.deconvolve(filtered, gauss)
n_q = len(sig) - len(gauss) + 1                      # the quotient has N - M + 1 samples ...
s_ = (len(sig) - n_q)//2                             # ... so pad it back to N, centered
deconv_res = np.zeros(len(sig)); deconv_res[s_:len(sig) - s_ - 1] = deconv
deconv = deconv_res*np.sum(gauss)
spec = lambda a: np.fft.fftshift(np.abs(np.fft.fft(a, nt)))/(3*nt)*nt/len(a)
fax = np.fft.fftshift(np.fft.fftfreq(nt))
rows = [(sig, C['input'], 'Box'), (gauss, C['impulse_resp'], 'Gaussian filter'), (filtered, C['output'], 'Box $\\ast$ Gaussian'),
        (deconv, C['phase'], 'Deconvolved result')]
fig, axs = plt.subplots(4, 2, figsize=(12, 10))
for i, (a_, col, name) in enumerate(rows):
    tl = tline2 if a_ is gauss else tline1
    axs[i, 0].plot(tl, a_, color=col, lw=2); axs[i, 0].set_xlim(0, nt if a_ is not gauss else nt//6)
    axs[i, 0].set_ylim(-0.25*np.max(a_), 1.15*np.max(a_)); axs[i, 0].set_xlabel('sample'); axs[i, 0].set_ylabel('amplitude')
    axs[i, 0].set_title(f'({chr(97 + 2*i)}) {name}: time domain', fontsize=12)
    S = spec(a_)
    axs[i, 1].plot(fax, S, color=col, lw=2); axs[i, 1].set_xlim(-0.5, 0.5); axs[i, 1].set_ylim(-0.05*S.max(), 1.15*S.max())
    axs[i, 1].set_xlabel('frequency [cycles/sample]'); axs[i, 1].set_ylabel('magnitude')
    axs[i, 1].set_title(f'({chr(98 + 2*i)}) {name}: frequency domain', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 7.14.</b> Convolution and deconvolution in the time (left) and frequency (right) domains. (a) A box and (b) its magnitude spectrum. (c) A Gaussian filter (with a small constant <code>DC_term</code> and a little random noise added) and (d) its spectrum. (e) The convolution of (a) and (c) and (f) its spectrum, the product of (b) and (d). (g) The result of deconvolving (c) from (e) with <code>scipy.signal.deconvolve</code>, which recovers the box almost exactly (relative RMS error 4e-08 away from the edges), and (h) its spectrum. <i>(Python-generated.)</i>

Overall, this process gives a fairly stable result.  However, if you play around with the parameters and, e.g., set *DC_term*=0 in the code section above, you'll see that things become unstable mighty quickly!

<a name="7.9.1"></a>
### 7.9.1 Deconvolution in practice

So far "decon" is an interesting theoretical operation ... but does it actually matter in practice? The answer is a resounding yes! Because we are using seismic waves to interrogate the Earth to learn more about its structure, it's very important that we attempt to remove as much of the wavelet effect as possible (which broadens the response and reduces the bandwidth, the range of frequencies present). Figure 7.15 shows a synthetic example, and <a href="https://csegrecorder.com/articles/view/deconvolution-after-migration">Bancroft et al. (2012) in the CSEG Recorder</a> shows the same improvement on real land data.

In [ ]:
nt, ntr, dt = 500, 60, 0.002                       # 1 s traces, 60 traces, 2 ms sampling
rng = np.random.default_rng(7)
refl = np.zeros((nt, ntr))
for t0, dip, amp in [(0.20, 0.0010, 1.0), (0.32, 0.0008, -0.7), (0.36, 0.0008, 0.6), (0.55, 0.0012, 0.8), (0.60, 0.0012, -0.5), (0.78, 0.0005, 0.7)]:
    for j in range(ntr):
        refl[int(round((t0 + dip*j)/dt)), j] += amp        # gently dipping layers
refl += 0.15*rng.standard_normal((nt, ntr))*(rng.random((nt, ntr)) < 0.03)   # sparse random reflectivity
tw = np.arange(0, 0.2, dt)
wav = np.exp(-tw/0.04)*np.sin(2*np.pi*25*tw)               # causal, front-loaded (minimum-phase-like) 25 Hz wavelet
data = np.array([np.convolve(refl[:, j], wav)[:nt] for j in range(ntr)]).T
data += 0.01*np.abs(data).max()*rng.standard_normal(data.shape)
Wf = np.fft.rfft(wav, nt); Df = np.fft.rfft(data, axis=0)
eps = 0.03*np.abs(Wf).max()                                # water level
dec = np.fft.irfft(Df*np.conj(Wf)[:, None]/np.maximum(np.abs(Wf)**2, eps**2)[:, None], nt, axis=0)
fr = np.fft.rfftfreq(nt, dt)
bp = np.clip((fr - 3)/5, 0, 1)*np.clip((100 - fr)/20, 0, 1)   # 3-100 Hz band-pass with linear tapers (keeps the whitened noise in check)
dec = np.fft.irfft(np.fft.rfft(dec, axis=0)*bp[:, None], nt, axis=0)
def bw20(a):
    S = np.abs(np.fft.rfft(a, axis=0)).mean(axis=1)
    above = fr[S >= 0.1*S.max()]
    return above.max() - above.min()
bw_before, bw_after = bw20(data), bw20(dec)

C = DSP_COLORS
tt = np.arange(nt)*dt
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for ax, a_, ttl in ((axs[0], data, '(a) Before deconvolution'), (axs[1], dec, '(b) After deconvolution')):
    clip = np.percentile(np.abs(a_), 99)
    ax.imshow(a_, cmap='gray_r', aspect='auto', vmin=-clip, vmax=clip, extent=[0.5, ntr + 0.5, tt[-1], 0])
    ax.set_xlabel('trace number'); ax.set_title(ttl, fontsize=12); ax.grid(False)
axs[0].set_ylabel('time [s]')
plt.tight_layout(); plt.show()
print(f'Bandwidth (-20 dB) before: {bw_before:.0f} Hz; after: {bw_after:.0f} Hz')

<b>Figure 7.15.</b> A synthetic seismic section (a) before and (b) after deconvolution. The 60 traces are a sparse, gently dipping reflectivity convolved with a causal, ringing 25 Hz wavelet, plus 1% noise. The deconvolution divides each trace's spectrum by the wavelet spectrum, with a water level of 3% of its maximum, and is followed by a 3–100 Hz band-pass filter. The reflectors become sharper and closely spaced pairs separate: the bandwidth at −20 dB grows from 53 Hz to 89 Hz. <i>(Python-generated.)</i>

<a name="7.10"></a>
## 7.10 Summary

An LTI system is completely described by its impulse response, and its output is the convolution of the input with that response. Convolution in time is multiplication in frequency, which explains why complex exponentials pass through LTI systems unchanged in shape, and why long convolutions are computed with the FFT. Crosscorrelation, a convolution with a time-reversed signal, measures delays and similarity, and the transform of an autocorrelation is the power spectrum. Deconvolution tries to undo convolution by spectral division, and it is only as stable as the smallest values of the spectrum it divides by.

## References

1. Bracewell, R.N., 1965, The Fourier Transform and Applications, McGraw-Hill, New York.
2. James, J.F. 2011, A Student's Guide to Fourier Transforms, 3rd ed, Cambridge University Press.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 7.1 &mdash; How wide is the recorded pulse?</b>

A seismic source wavelet is approximated by the Gaussian $w(t) = e^{-t^2/a^2}$ with $a = 10$ ms. The geophone and recording system together have the Gaussian impulse response $g(t) = e^{-t^2/b^2}$ with $b = 7.5$ ms. The recorded pulse is $p = w\ast g$, and two reflections of equal strength arrive $\Delta = 10$ ms apart.

<b>(a)</b> Using Section 7.6.3, what is the width parameter $c$ of the recorded pulse $p(t) \propto e^{-t^2/c^2}$?

<b>(b)</b> Using the Gaussian pair of Module 5 and the convolution theorem, find the frequency $f_e$ at which each spectrum has fallen to $1/e$ of its peak, for the source alone and for the recorded pulse (MATH 348: Fourier transform of a Gaussian).

<b>(c)</b> A colleague argues: "The source pulse is 10 ms wide, so two reflections 10 ms apart will appear as two separate peaks." Test this claim. (Hint: the sum of two Gaussians of width $c$ separated by $\Delta$ has a dip at the midpoint only if $\Delta > \sqrt{2}\,c$.) Which figure in this module shows the same effect?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 7.1</b></summary>

<b>(a)</b> Equation (29): $c = \sqrt{a^2 + b^2} = \sqrt{100 + 56.25} = 12.5$ ms (a numerical convolution gives 12.495 ms).

<b>(b)</b> The spectrum of $e^{-t^2/c^2}$ is proportional to $e^{-c^2\omega^2/4}$, so it falls to $1/e$ at $\omega = 2/c$, i.e. $f_e = 1/(\pi c)$. Source: $1/(\pi \times 0.010) = 31.83$ Hz; recorded: $1/(\pi \times 0.0125) = 25.46$ Hz. Convolution multiplied the spectra, so the bandwidth can only shrink.

<b>(c)</b> $\sqrt{2}\,c = 17.68$ ms $> 10$ ms, so the two reflections merge into <b>one</b> peak. The source width is the wrong yardstick: the <i>recorded</i> width sets the resolution, and the instrument made it worse. This is the interference in Figure 7.4(h), and it is why deconvolution (Section 7.9) is worth the trouble.
</details>

<div class="alert alert-warning">
<b>Microquestion 7.2 &mdash; An RC circuit as an LTI system</b>

A PHGN 200 RC low-pass filter with $R = 1$ k$\Omega$ and $C = 1$ $\mu$F ($\tau = RC = 1$ ms) has the impulse response $h(t) = (1/\tau)e^{-t/\tau}u(t)$. A switch applies the step $x(t) = u(t)$ at $t = 0$. A student samples $h$ at $T = 0.05$ ms for $0 \le t \le 5$ ms ($L = 101$ samples), places the step at sample 100 of a 200-sample array, and plots <code>np.convolve(x, h, mode='same')</code>.

<b>(a)</b> Evaluate the convolution integral, equation (4), to find $y(t)$, and give $y(2\ \mathrm{ms})$.

<b>(b)</b> Use the convolution theorem and the derivative theorem, equation (32), on $\tau\,dy/dt + y = x$ to find $\widehat{H}(\omega)$. Show that $|\widehat{H}| = 1/\sqrt{2}$ at the PHGN 200 corner frequency, and give $f_c$.

<b>(c)</b> The student's plot shows the output starting <b>2.5 ms before</b> the switch closes. Is the circuit violating physics? Explain what <code>mode='same'</code> did, and give the fix.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 7.2</b></summary>

<b>(a)</b> $y(t) = \int_0^t (1/\tau)\,e^{-(t-s)/\tau}\,ds = 1 - e^{-t/\tau}$ for $t \ge 0$ (MATH 225: the same result as the first-order ODE). $y(2\ \mathrm{ms}) = 1 - e^{-2} = 0.8647$.

<b>(b)</b> Transforming gives $(i\omega\tau + 1)\widehat{Y} = \widehat{X}$, so $\widehat{H}(\omega) = 1/(1 + i\omega\tau)$. At $\omega\tau = 1$, $|\widehat{H}| = 1/\sqrt{2} = 0.7071$ (phase $-45^\circ$), with $f_c = 1/(2\pi\tau) = 159.15$ Hz.

<b>(c)</b> No. <code>mode='same'</code> returns the central $N$ samples of the full $(N + L - 1)$-sample result, so for an $L = 101$ kernel it drops the first $(L - 1)/2 = 50$ samples $= 2.5$ ms. Everything appears 2.5 ms early: the first nonzero output is at index 50 instead of 100. A causal kernel needs <code>mode='full'</code>, keeping the first $N$ samples (first nonzero output at index 100), or a zero-phase kernel centered in the array, as in Figure 7.3.
</details>

<div class="alert alert-warning">
<b>Microquestion 7.3 &mdash; Picking a GPR reflection by crosscorrelation</b>

A GPR antenna emits $x(t) = e^{-t^2/a^2}$ with $a = 2$ ns. The trace contains one reflection, $y(t) = 0.4\,x(t - t_0)$. The ground has relative permittivity $\varepsilon_r = 9$.

<b>(a)</b> Using definition (50), show that $(x\star y)(\tau)$ peaks at $\tau = t_0$, and find the peak value as a multiple of the autocorrelation peak $a\sqrt{\pi/2}$.

<b>(b)</b> The peak occurs at $t_0 = 60$ ns. Using the PHGN 200 EM wave speed $v = c/\sqrt{\varepsilon_r}$, find the reflector depth ($t_0$ is a two-way time). Using the autocorrelation of a Gaussian, what is the width parameter of the correlation peak?

<b>(c)</b> A student computes <code>np.correlate(x, y, 'full')</code> and reports the reflection at $-60$ ns. What went wrong, and which call matches equation (50)?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 7.3</b></summary>

<b>(a)</b> $(x\star y)(\tau) = 0.4\int x(t)\,x(t + \tau - t_0)\,dt = 0.4\,a_x(\tau - t_0)$, which peaks at $\tau = t_0$ with value $0.4\,a\sqrt{\pi/2} = 0.4 \times 2.5066 = 1.0027$ (amplitude$^2$ ns).

<b>(b)</b> $v = 0.2998/3 = 0.0999$ m/ns $\approx 0.1$ m/ns, so the depth is $v t_0/2 = 3.00$ m (PHGN 100: distance = speed × time, halved for the round trip). The autocorrelation of $e^{-t^2/a^2}$ is $a\sqrt{\pi/2}\,e^{-\tau^2/(2a^2)}$, so the width parameter of the peak is $a\sqrt{2} = 2.83$ ns.

<b>(c)</b> NumPy conjugates and slides its <b>second</b> argument, so <code>np.correlate(x, y)</code> computes $(y\star x)(\tau) = (x\star y)(-\tau)$, which peaks at $-60$ ns. <code>np.correlate(y, x, 'full')</code> matches equation (50) and peaks at $+60$ ns. Figure 7.12(c) shows the same reversal, and equation (51) explains it.
</details>